# Lab 1: weather data, climatologies, and the persistence forecast

This lab builds familiarity with ERA5 as an xarray object: how the reanalysis is stored, how a climatology and an anomaly are computed from it, how much a field changes from one time step to the next, and how a forecast is verified against it.
The forecast we verify is the simplest one, persistence, and the verification code you write for it is the code that verifies every model you train later.

The notebook asks questions and names the objects each answer produces (their name, dimensions, and units), and it gives no solution code.
Work through the sections in order, since each opens on an object the previous one produced.
Every plot and number in it is asked for a reason: on each one, say why you are looking at it and what it tells you, rather than only producing it.

Keep the objects you build (`data`, the climatologies, the statistics, and the verification functions), because the next lab starts from them.

## Setup

The imports below are the whole toolkit for this lab.
xarray carries the data with named dimensions and coordinates, and inside it every transpose and reduction is by name already.
numpy is for the moments where you leave xarray, einops for naming the axes of a numpy array or a tensor when you do, and matplotlib is used through xarray's own `.plot` methods wherever it can be.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from einops import rearrange, reduce, repeat

# bottleneck accumulates float32 sums in float32 and is off by tens of kelvin on a five-year mean;
# without it xarray falls back to numpy's pairwise summation
xr.set_options(keep_attrs=True, display_expand_data=False, use_bottleneck=False)

### Documentation

xarray

- [Dataset](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.html) and [DataArray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.html): the two objects you work with here.
- [Indexing and selecting](https://docs.xarray.dev/en/stable/user-guide/indexing.html): `.sel` by coordinate value and `.isel` by position.
- [groupby](https://docs.xarray.dev/en/stable/user-guide/groupby.html): split, apply, and combine over a coordinate such as `time.dayofyear`.
- [diff](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.diff.html): the difference along one dimension.
- [rolling](https://docs.xarray.dev/en/stable/user-guide/computation.html#rolling-window-operations): windowed reductions along one dimension.
- [Plotting](https://docs.xarray.dev/en/stable/user-guide/plotting.html): `.plot()` picks a line, a histogram, or a map from the dimensions of the array.

Data

- [WeatherBench 2 data guide](https://weatherbench2.readthedocs.io/en/latest/data-guide.html): every store below, its grid, and its variables.
- [ARCO-ERA5](https://github.com/google-research/arco-era5): the native-resolution ERA5 archive that is updated continuously.

### Data stores

All data live in public Google Cloud buckets and open with `xr.open_zarr` and anonymous access.
Opening is lazy: nothing downloads until you `.load()`, `.plot()`, or read `.values`.

- `era5_5p6` and `era5_1p5`: ERA5 1959 to 2023-01-10 at 6-hourly cadence, on a 64 by 32 grid (5.625 degrees) and a 240 by 121 grid (1.5 degrees), 13 pressure levels.
- `clim_5p6` and `clim_1p5`: the WeatherBench 2 climatology on the same two grids, 1990 to 2019, with dimensions `hour` and `dayofyear`.
  At 5.625 degrees one chunk is the whole `(hour, dayofyear)` block of a variable, and for a level variable the whole column, so one level of the climatology costs about 150 MB of download; take it once.
- `hres_5p6` and `hres_1p5`: the ECMWF HRES forecasts 2016 to 2022 initialised at 00 and 12 UTC, with dimensions `time` (initialisation) and `prediction_timedelta` (lead) out to 10 days.
- `arco`: ERA5 at its native 0.25 degree, hourly, 37 levels, updated to about a week ago; the recent week is the only part of it we use, and section 7 says why.
  Each hour is one chunk of the whole globe (about 2 MB compressed, 4 MB in memory), so a week of one variable is 168 chunks, and a time series at one point costs the same download as the globe.

The unit of download is the store's chunk.
At 5.625 degrees a chunk holds 100 time steps and, for a variable with levels, all 13 levels, so selecting one level costs the same download as the whole column.
Five years of the nine variables below is about 550 MB in memory, and about 3 GB of download when the level variables come with their full column.
Load once, cache what you loaded (`to_netcdf` or `to_zarr`), and reopen the cache after that.
The 1.5 degree store is 14 times larger per field; keep it lazy and select before you load.

In [ ]:
ANON = {"token": "anon"}

STORES = {
    "era5_5p6": "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr",
    "era5_1p5": "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr",
    "clim_5p6": "gs://weatherbench2/datasets/era5-hourly-climatology/1990-2019_6h_64x32_equiangular_conservative.zarr",
    "clim_1p5": "gs://weatherbench2/datasets/era5-hourly-climatology/1990-2019_6h_240x121_equiangular_with_poles_conservative.zarr",
    "hres_5p6": "gs://weatherbench2/datasets/hres/2016-2022-0012-64x32_equiangular_conservative.zarr",
    "hres_1p5": "gs://weatherbench2/datasets/hres/2016-2022-0012-240x121_equiangular_with_poles_conservative.zarr",
    "arco": "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
}

era5 = xr.open_zarr(STORES["era5_5p6"], storage_options=ANON, chunks={})
clim = xr.open_zarr(STORES["clim_5p6"], storage_options=ANON, chunks={})
era5

### Variables

The short names on the left are the ones we use throughout; the right-hand side is the WeatherBench 2 variable name and the pressure level, `None` for a surface variable.
The order of this dictionary is the channel order of every array you build from it.

In [ ]:
VARIABLES = {
    "T2M":  ("2m_temperature", None),           # K
    "U10M": ("10m_u_component_of_wind", None),  # m/s
    "V10M": ("10m_v_component_of_wind", None),  # m/s
    "TP6h": ("total_precipitation_6hr", None),  # m, accumulated over the six hours ending at the time stamp
    "Z500": ("geopotential", 500),              # m^2/s^2
    "T850": ("temperature", 850),               # K
    "Q700": ("specific_humidity", 700),         # kg/kg
    "U250": ("u_component_of_wind", 250),       # m/s
    "V250": ("v_component_of_wind", 250),       # m/s
}

PERIOD = slice("2015", "2019")   # five years, inclusive
AUGSBURG = dict(latitude=48.37, longitude=10.90)

## 1. What is in the store, and how do we look at it?

**1.1**
What does the repr of `era5` tell you about the size of the store, its coordinates, and its chunking?
Which variables carry a `level` dimension, in which order do the dimensions of a variable come, and how are the latitude and longitude coordinates ordered and spaced?
Where does the longitude coordinate start and end?

**1.2**
How do we build `data`, a Dataset with the nine variables of `VARIABLES` as data variables named by their short keys, over `PERIOD`, with dimensions `(time, latitude, longitude)` and loaded into memory?
Which xarray call selects a period from year strings, which selects a level and drops the coordinate, and which puts the dimensions in the order named above?
How large is `data` in memory (`data.nbytes`), how long did the load take, and where do you save it so that you never load it twice?

**1.3**
What do a `T2M` and a `Z500` field look like at one time step, and what do the units in the attributes say?
Which large-scale structure is visible in each?

**1.4**
How do we crop?
`.sel` selects by coordinate value and `.isel` by position: select one year, one day, the 00 UTC time steps only, and Europe (35 to 72 degrees north, 25 degrees west to 45 degrees east).
Look at the longitude coordinate of your Europe crop next to the longitude coordinate of the store, and say why the crop needs care where the coordinate wraps.
Plot a European `T2M` map for one day of the 2019 heat wave (25 July 2019).

**1.5**
What is the histogram of all values of one variable, over the whole period and the whole grid?
Compare `T2M` with `TP6h` (a logarithmic count axis helps).
What does the shape of each histogram say about how the variable is distributed?

**1.6**
What does a vertical profile look like?
From `era5["temperature"]` (and `era5["geopotential"]`), take one time step at the grid point nearest Augsburg (`AUGSBURG`, with `method="nearest"`) and plot the value against the 13 pressure levels from 50 to 1000 hPa, with pressure on the vertical axis and increasing downward.
Where is the tropopause?

**1.7**
What is the time series of `T2M` at the grid point nearest Augsburg over the five years, and over one summer week?
Which cycles are visible at each scale?

## 2. What is the expected weather, and how far from it are we?

**2.1**
How do we compute a day-of-year climatology from `data`?
The target is `clim_doy`, with dimensions `(dayofyear, latitude, longitude)` and the units of the variable.
How do we compute an hour-of-day climatology, `clim_hod` with dimension `(hour, latitude, longitude)`, and a climatology over both, `clim_doy_hod` with `(hour, dayofyear, latitude, longitude)` like the WeatherBench 2 store?
Plot all three at Augsburg for `T2M`.
How does the WeatherBench 2 climatology (`clim`) differ from yours at the same point, and why (which years, and what smoothing does its documentation describe)?

**2.2**
How do we compute the anomaly of `data` with respect to each climatology?
The anomaly is $x' = x - c$, where $c$ is the climatological value for the time step's day of year (or hour of day) at that grid point.
The target is `anom_doy`, with the dimensions and units of `data`, and the same for `anom_hod`.
Plot the anomaly time series at Augsburg for the summer of 2019 and a map of the `T2M` anomaly on 25 July 2019.
Which anomaly is larger, that against the day-of-year climatology or against the hour-of-day one, and why?

**2.3**
What is the activity of the data?
Activity is the size of the anomaly, $a = \sqrt{\langle x'^2 \rangle}$, with the mean taken over time for the activity of a grid point (`(latitude, longitude)` per variable) and over time and, area-weighted, over grid points for the activity of a variable (one number).
Which variables and which regions are most active?
Keep the activity of every variable: it is the level a climatological forecast's error sits at, and the persistence curves of section 5 are read against it.

**2.4**
Which statistics normalise the data?
Standardisation replaces $x$ by $(x - \mu)/\sigma$; compute the pointwise mean and standard deviation of every variable over time (`(latitude, longitude)` per variable) and the global mean and standard deviation over `(time, latitude, longitude)` (one number per variable).
Which of the two would you use to standardise a model input, and what does each hide?
Save them as a Dataset with a `statistic` dimension, since the Dataset class of the next lab reads them.

## 3. How much does the atmosphere change in one time step?

**3.1**
What is the tendency of the data?
The tendency over one step is $\delta x(t) = x(t + 6\,\mathrm{h}) - x(t)$, in the units of the variable, with one time step fewer than the data.
Compute the mean and the standard deviation of the tendency over time, per grid point, for every variable in `data`.
Plot the histogram of the pointwise standard deviations, and the histogram of the pointwise means, one panel per variable.
Which variables change most in one step relative to their own activity from section 2, and which change least?

**3.2**
How does the tendency depend on the level?
Take one year of `temperature`, `geopotential`, `u_component_of_wind`, and `specific_humidity` at all 13 levels from `era5`, compute the same two statistics of the six-hour difference, and plot their histograms per level.
Where in the column does each variable change most, and does the tendency scale with the level's own standard deviation?
Keep the tendency mean and standard deviation of every variable in `data` alongside the statistics of section 2, because the model of the next lab can predict either the next state or the tendency, and each target has its own normalisation.

## 4. How do we get from xarray to numpy and back?

**4.1**
How do we turn `data` into one numpy array `array` of shape `(time, channel, latitude, longitude)`, where the channel axis is ordered as `VARIABLES`?
How do you make the channel order follow the dictionary rather than the Dataset's own order?
Inside xarray the dimension names order the axes; once you hold a bare numpy array, name the axes of any rearrangement with einops rather than positional indices.

**4.2**
How do we turn `array` back into a Dataset `data_back` with the same variables, dimensions, coordinates, and units as `data`?
Assert that the round trip is exact.
What would go wrong silently if the channel order in one direction differed from the other?

## 5. How well does persistence forecast?

The persistence forecast for lead time $\tau$ from initialisation $t$ is the state at $t$.
WeatherBench 2 indexes a forecast by its initialisation `time` and its lead `prediction_timedelta`, and its valid time is the sum of the two; open `STORES["hres_5p6"]` and read the dimensions, coordinates, and lead times of a real forecast archive.

**5.1**
How do we build `persistence`, a Dataset with dimensions `(time, prediction_timedelta, latitude, longitude)`, `prediction_timedelta` from 0 to 240 hours in steps of 6 hours, whose value at every lead is the state at the initialisation time?
Five years of initialisations at 41 leads is 41 times the memory of `data`, so choose the initialisations of 2019 at 00 UTC and the variables `Z500` and `T2M`.

**5.2**
How do we align the truth?
The target is `truth`, with the same dimensions and coordinates as `persistence`, holding the ERA5 state at the valid time `time + prediction_timedelta`.
What happens at the end of 2019, and how do you deal with it?

**5.3**
How large is a grid cell?
On a regular latitude-longitude grid the cells narrow toward the poles, so a plain mean over grid points weights the poles too heavily.
The area of the cell centred at latitude $\varphi_i$ with spacing $\Delta\varphi$ and $\Delta\lambda$ (in radians) is $A_i = R^2 \,\Delta\lambda \,\big(\sin(\varphi_i + \tfrac{\Delta\varphi}{2}) - \sin(\varphi_i - \tfrac{\Delta\varphi}{2})\big)$ with $R = 6371$ km.
Compute `cell_area`, with dimension `(latitude)` and units of km², and check that it sums over the grid to the surface of the Earth (510 million km²).
The area weight used in verification is $w_i = \cos\varphi_i / \overline{\cos\varphi}$, normalised to mean one, so that a weighted mean $\langle x \rangle_w = \sum_i w_i x_i / \sum_i w_i$ reduces to the plain mean on an equal-area grid.
Compare $w_i$ with `cell_area` normalised the same way: on which of the two grids (5.625 degrees, latitudes at cell centres; 1.5 degrees, rows on the poles) are they identical, and why?

**5.4**
How do we score?
Write functions of `(forecast, truth, climatology)` that return, per lead time and variable,

- the root-mean-square error, $\mathrm{RMSE} = \sqrt{\langle (f - t)^2 \rangle_w}$ over the grid, then averaged over initialisations,
- the anomaly correlation, $\mathrm{ACC} = \langle f' t' \rangle_w \,/\, \sqrt{\langle f'^2 \rangle_w \langle t'^2 \rangle_w}$, with anomalies $f' = f - c$ and $t' = t - c$ against a climatology (which one, yours from 2015 to 2019 or the WeatherBench 2 one from 1990 to 2019, and why does the choice matter?),
- the activity of the forecast and of the truth, $a_f = \sqrt{\langle f'^2 \rangle_w}$ and $a_t = \sqrt{\langle t'^2 \rangle_w}$.

Compute the RMSE once with the area weights and once without, and plot both against lead time for `Z500` and `T2M`.
How large is the difference, and which way does the unweighted error go?
Then plot the RMSE and the ACC against lead time.
Where do the RMSE curves start, where do they saturate, and how do those levels relate to the activity from section 2?
Look at the shape of the `T2M` curves next to the `Z500` curves, and explain the difference.

**5.5**
How well does climatology forecast?
The climatological forecast for any initialisation and lead is the climatological value at the valid time, so it has the dimensions of `persistence` and does not depend on the initial state.
Build `climatology_forecast` from the climatology of section 2 and score it with the functions of 5.4.
Where does its RMSE sit against the activity of section 2 and against the persistence curve, and at which lead time do persistence and climatology cross for `Z500` and for `T2M`?
What is the anomaly correlation of the climatological forecast, and why?

**5.6**
How much better than a reference is a forecast?
The skill score of a forecast against a reference forecast is $\mathrm{SS} = 1 - \mathrm{RMSE}_f / \mathrm{RMSE}_{\mathrm{ref}}$, so that a perfect forecast scores 1, the reference itself 0, and a forecast worse than the reference below 0; scorecards show skill scores, not raw errors, and the reference is named with them.
Compute the skill score of persistence against climatology at every lead time and plot it for `Z500` and `T2M`.
Where does it cross zero, what does a lead time beyond the crossing mean for persistence as a forecast, and what would the skill score of climatology against persistence look like?

**5.7**
How do we separate error along the true anomaly from error orthogonal to it?
Let $a_f$ and $a_t$ be the activities of forecast and truth.
The information is $p = a_f\,\mathrm{ACC}$, the information error is $\mathrm{IE} = |a_t - p|$, and the noise error is $\mathrm{NE} = \sqrt{\mathrm{RMSE}^2 - \mathrm{IE}^2}$ (Bonavita and Geer, QJRMS 2026).
Compute the two for persistence at every lead and draw the information, noise, and correlation diagram: the helper below draws the axes and the reference arcs, and you pass it your numbers.
Where does persistence sit on the diagram at short leads, and where does it move as the lead grows?
Which of its two errors grows?

**5.8**
How does a real forecast compare?
`STORES["hres_5p6"]` holds HRES for the same initialisations and leads; select 2019 at 00 UTC, verify it with the same functions against the same truth, and add it to the curves and to the diagram.
The archive is chunked in four initialisations by one lead with the full column, so a year of one level variable is about 1.5 GB of download; start with `T2M` and one month.
Which truth does WeatherBench 2 verify HRES against, and what changes if you use ERA5 instead?

In [ ]:
def information_noise_diagram(ax, noise_error, information, true_activity, label=None, **plot_kwargs):
    '''Information, noise, and correlation diagram after Bonavita and Geer (2026), Figure 3.

    A forecast is a point at (noise error, information).
    Its distance from the origin is the forecast activity, the angle to the vertical axis
    has cosine equal to the anomaly correlation, and its distance to the point (0, true_activity)
    is the error.  The dashed quarter circle is the locus of forecasts with the true activity;
    the dashed half circle is the locus of lowest error for a given correlation.
    '''
    a = float(true_activity)
    theta = np.linspace(0, np.pi / 2, 200)
    ax.plot(a * np.sin(theta), a * np.cos(theta), "k--", lw=0.8)                       # forecast activity == true activity
    ax.plot(a / 2 * np.sin(2 * theta), a / 2 + a / 2 * np.cos(2 * theta), "k:", lw=0.8)  # lowest error for a given correlation
    for acc in (0.9, 0.8, 0.6, 0.3):
        ang = np.arccos(acc)
        ax.plot([0, 1.1 * a * np.sin(ang)], [0, 1.1 * a * np.cos(ang)], color="0.7", lw=0.6)
        ax.annotate(f"ACC {acc}", (1.12 * a * np.sin(ang), 1.12 * a * np.cos(ang)), fontsize=7, color="0.4")
    ax.plot(0, a, "k*", ms=9)                                                          # the perfect forecast
    ax.plot(noise_error, information, "o-", label=label, **plot_kwargs)
    ax.set_aspect("equal")
    ax.set_xlim(0, 1.25 * a)
    ax.set_ylim(0, 1.25 * a)
    ax.set_xlabel("noise error")
    ax.set_ylabel("information")
    return ax


# a demonstration with made-up numbers: a forecast that loses correlation with lead time
fig, ax = plt.subplots(figsize=(4, 4))
acc = np.array([0.98, 0.9, 0.75, 0.55, 0.3])
a_f, a_t = 1.0, 1.0
information_noise_diagram(ax, a_f * np.sqrt(1 - acc**2), a_f * acc, a_t, label="a made-up forecast")
ax.legend(loc="lower right")
plt.show()

## 6. Case study: can we find a hurricane in the pressure field?

Hurricane Dorian reached category 5 over the Bahamas on 1 September 2019, stalled there for two days, and then moved north along the coast of the United States.

**6.1**
How do we grab that week?
Take `mean_sea_level_pressure` from `STORES["era5_1p5"]` for 30 August to 6 September 2019 in the box 10 to 50 degrees north and 100 to 40 degrees west, and load it.
Plot the field at 12 UTC on 1 September: is the storm visible, and how deep is the minimum compared with the observed 910 hPa?
Which coordinate does `.plot()` put on the horizontal axis, and why?

**6.2**
How do we find the track?
The location of the pressure minimum in the box, at every time step, is a first estimate of the storm centre.
Plot the track over one snapshot of the field, and look at the whole of it before you trust it: is every point of it the storm, how do you know, and what would you add to the estimate?

**6.3**
What does the storm look like at native resolution?
Take the same week from `STORES["arco"]` at 0.25 degrees (every sixth hour): how do the minimum pressure and the track compare with the 1.5 degree version, and why?

## 7. What did the atmosphere do last week?

The WeatherBench 2 stores end on 10 January 2023.
The ARCO-ERA5 store is updated continuously and carries two stop dates in its attributes: `valid_time_stop` for final ERA5, and `valid_time_stop_era5t` for the preliminary ERA5T that runs about six days behind real time.
Its `time` axis is padded well beyond the data, so read the attributes before you select.

**7.1**
Which is the most recent day in the store, and how do we select the last seven days of `2m_temperature` at Augsburg?
Plot the time series (hourly, this time; take every sixth hour if the connection is slow, since every hour is a full-globe chunk) and the map of the most recent day over Europe.

**7.2**
How warm was that week compared with the expectation?
The 1990 to 2019 climatology exists at 1.5 degrees and the ARCO field at 0.25 degrees; which of the two do you move onto the other's grid, how, and what does the anomaly of the last week look like?

## 8. Deeper: where does the variance live in space?

**8.1**
What is the power spectrum of a field along longitude?
For one latitude band and one time step, take the discrete Fourier transform of `Z500` along `longitude` (`numpy.fft.rfft`), and plot the power against the zonal wavenumber on logarithmic axes.
Compare `Z500`, `T2M`, `U250`, and `TP6h`: which decays fastest with wavenumber, and which is flattest?

**8.2**
What is the radially averaged power spectral density?
Take the two-dimensional transform of a field over `(latitude, longitude)`, compute the radial wavenumber of every Fourier coefficient, and average the power in radial bins (`numpy.digitize` or `scipy.stats.binned_statistic`).
The result is one curve per field; plot it for the same four variables, and for the anomaly of one variable against its full field.
What does the shape of the curve say about how the variance is distributed across scales, and why is the persistence forecast's spectrum identical to the analysis at every lead time?